In [ ]:
!pip install langchain langchain-community langchain-openai faiss-cpu pypdf

In [ ]:
"""
Author: Aquiles Elbaum

Description:
    Retrieval-augmented generation pipeline for sensor recommendation using
    a FAISS vector store and Gemma, with attention matrix extraction.
"""

import os
import gc
from google.colab import drive

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

from faiss import IndexHNSWFlat
import numpy as np

# Load data
drive.mount('/content/drive')

pdf_folder = "/content/drive/MyDrive/Capstone/Data/Documents/C5ISR"

pdf_files = [
    os.path.join(pdf_folder, f)
    for f in os.listdir(pdf_folder)
    if f.lower().endswith(".pdf")
]

print("PDFs found:", len(pdf_files))
for f in pdf_files:
    print(" -", os.path.basename(f))

# Split all pdfs into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)

all_chunks = []

counter = 0
for pdf_path in pdf_files:
    counter++
    print(f"\nProcessing PDF {counter} / {len(pdf_files)}: {os.path.basename(pdf_path)}")

    loader = PyPDFLoader(pdf_path, mode="single")
    docs = loader.load()

    splits = text_splitter.split_documents(docs)

    for chunk in splits:
        chunk.metadata["source"] = os.path.basename(pdf_path)

    all_chunks.extend(splits)
    gc.collect()

print(f"\nTotal text chunks: {len(all_chunks)}")

# Use an embedding model to embed all text chunks
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 16}
)

texts = [doc.page_content for doc in all_chunks]
metadatas = [doc.metadata for doc in all_chunks]

print("Embedding chunks...")
vectors = embeddings.embed_documents(texts)
vectors = np.array(vectors).astype("float32")

d = 384  # MiniLM dimension
hnsw_index = IndexHNSWFlat(d, 32)
hnsw_index.hnsw.efConstruction = 40

# Save embedded data vectorstore along with the original text chunks
print("Adding vectors to HNSW index...")
hnsw_index.add(vectors)

print("Building vectorstore...")
docstore = InMemoryDocstore()
index_to_docstore_id = {}

for i, doc in enumerate(all_chunks):
    doc_id = str(i)
    docstore.add({doc_id: doc})
    index_to_docstore_id[i] = doc_id

vectorstore = FAISS(
    embedding_function=embeddings,
    index=hnsw_index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id
)


save_path = "/content/drive/MyDrive/Capstone/RAG_Data/Vectorstore_Final"
vectorstore.save_local(save_path)

print("\nVectorstore saved to:", save_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PDFs found: 32
 - Buried-Object-Detection Improvements Incorporating Environmental Phenomenology into Signature Physics.pdf
 - ThermalInfraredComparisonStudyofBuriedObjects.pdf
 - ModelingofaMulti-MonthThermalIRStudy.pdf
 - ModernizingEnvironmentalSignaturePhysicsforTargetDetection.pdf
 - EnvironmentallyInformedBuriedObjectRecognition.pdf
 - GuidelinesforAtmosphericMeasurementsinSupportofElectroOptical.pdf
 - ClutterCharacterizationforDownLookingGroundPenetratingRadar.pdf
 - PerformanceAnalysisofSideLookingGroundPenetratingRadarImaging.pdf
 - PrinciplesOfModernRadarTextbook.pdf
 - ImagingStudyforSmallUnmannedAerialVehicleUAVMountedGPR.pdf
 - Comparing the Thermal Infrared Signatures of Shallow Buried Objects and Distrubed Soil.pdf
 - IMPACT OF SOIL WATER CONTENT ON LANDMINE DETECTION.pdf
 - EXPERIMENTAL RESULTS OF GROUND DISTURBANCE DETECTION USING UNCOOLED I